# Advanced 08 — Agent Identity Lifecycle, Governance & Operational Excellence

**Scenario:** your enterprise has moved from 20 experimental agents to 1,200+ agent-related identities across Kubernetes, cloud IAM, OAuth, MCP, CI/CD, external partners and agent platforms. Build the governance control plane that discovers, approves, provisions, monitors, reviews and retires them.

The notebook intentionally treats governance as executable engineering rather than policy prose.


In [ ]:
from datetime import datetime,timedelta,timezone
from pydantic import BaseModel,Field
from enum import Enum
import pandas as pd, networkx as nx, json, uuid
NOW=datetime.now(timezone.utc)


## 1 — Inventory schema

In [ ]:
class AgentIdentity(BaseModel):
    id:str; name:str; identity_type:str; owner:str; purpose:str
    environment:str; risk_tier:str; status:str="draft"
    short_lived:bool=True; monitoring:bool=False; runtime_bound:bool=False
    review_interval_days:int=90; delegation_allowed:bool=False


## 2 — Identity taxonomy

In [ ]:
types=["logical-agent","deployment","workload","sub-agent","tool","mcp-server",
"oauth-client","cloud-role","external-agent","ci-cd"]
pd.DataFrame({"identity_type":types})


## 3 — Create inventory

In [ ]:
inventory=[
AgentIdentity(id="agt-001",name="Claims Assistant",identity_type="logical-agent",owner="claims-team",
 purpose="assist claims processing",environment="prod",risk_tier="high",monitoring=True,runtime_bound=True,review_interval_days=60),
AgentIdentity(id="wrk-005",name="Claims Runtime",identity_type="workload",owner="ai-platform",
 purpose="run claims agent",environment="prod",risk_tier="medium",monitoring=True,runtime_bound=True)]
pd.DataFrame([x.model_dump() for x in inventory])


## 4 — Ownership control

In [ ]:
def ownership_findings(items):
    return [x.id for x in items if not x.owner.strip()]
ownership_findings(inventory)


## 5 — Risk model

In [ ]:
weights={"autonomy":.15,"write":.2,"sensitive_data":.15,"external":.1,"delegation":.15,"financial":.15,"credential":.1}
def risk_score(values):return sum(values[k]*weights[k] for k in weights)
score=risk_score({"autonomy":.8,"write":.8,"sensitive_data":.9,"external":.3,"delegation":.2,"financial":.7,"credential":.2})
score


## 6 — Risk tier

In [ ]:
def tier(s):
    if s<.25:return "low"
    if s<.5:return "medium"
    if s<.75:return "high"
    return "critical"
tier(score)


## 7 — Lifecycle state machine

In [ ]:
allowed={"draft":{"pending_approval"},"pending_approval":{"approved","revoked"},
"approved":{"active","revoked"},"active":{"suspended","quarantined","retired","revoked"},
"suspended":{"active","retired","revoked"},"quarantined":{"suspended","revoked"},
"retired":{"revoked"},"revoked":set()}
def transition(current,target):
    if target not in allowed[current]:raise ValueError(f"blocked {current}->{target}")
    return target


## 8 — Prevent lifecycle bypass

In [ ]:
try: transition("draft","active")
except ValueError as e: print(e)


## 9 — Registration request

In [ ]:
registration={"id":"agt-101","owner":"underwriting-ai","purpose":"submission triage",
"environment":"prod","requested_actions":["submission.read"],"risk_tier":"medium",
"identity_mechanism":"SPIFFE","delegation":False}
registration


## 10 — Approval routing

In [ ]:
approval_policy={"low":{"owner"},"medium":{"owner","platform"},
"high":{"owner","security"},"critical":{"owner","security","risk","business_authority"}}
approval_policy[registration["risk_tier"]]


## 11 — Segregation of duties

In [ ]:
def sod(requester,approvers):return requester not in set(approvers)
sod("developer-a",["owner-b","security-c"]),sod("developer-a",["developer-a","security-c"])


## 12 — Identity manifest

In [ ]:
manifest={"agent":{"id":"agt-101","owner":"underwriting-ai","risk":"medium","environment":"prod"},
"identity":{"workload":"spiffe","static_secrets":False},
"authority":{"resources":["submissions"],"actions":["read"],"delegation":False},
"review":{"interval_days":90}}
manifest


## 13 — Policy-as-code gate

In [ ]:
def deployment_gate(m,approvals):
    return (m["agent"]["owner"] and not m["identity"]["static_secrets"]
            and set(approval_policy[m["agent"]["risk"]]).issubset(approvals))
deployment_gate(manifest,{"owner","platform"})


## 14 — CI/CD negative test

In [ ]:
bad=manifest.copy();bad={**manifest,"identity":{"workload":"spiffe","static_secrets":True}}
deployment_gate(bad,{"owner","platform"})


## 15 — Provisioning plan

In [ ]:
plan=["register logical identity","create SPIFFE mapping","create tool policy",
"enroll monitoring","schedule review"]
pd.DataFrame({"order":range(1,len(plan)+1),"operation":plan})


## 16 — Least privilege

In [ ]:
requested={"submission.read","submission.update","payment.approve"}
required={"submission.read"}
excess=requested-required
excess


## 17 — Delegation governance

In [ ]:
delegation_policy={"max_depth":1,"max_minutes":30,"allowed_actions":{"submission.read"}}
request={"depth":1,"minutes":15,"actions":{"submission.read"}}
request["depth"]<=1 and request["minutes"]<=30 and request["actions"].issubset(delegation_policy["allowed_actions"])


## 18 — Sub-agent registration

In [ ]:
subagent={"id":"sub-991","parent":"agt-101","owner":"underwriting-ai",
"expires":NOW+timedelta(minutes=20),"actions":["submission.read"]}
subagent


## 19 — Runtime binding

In [ ]:
approved={"agent":"agt-101","deployment":"sha256:abc","workload":"spiffe://prod.example/agents/underwriting"}
observed={"agent":"agt-101","deployment":"sha256:abc","workload":"spiffe://prod.example/agents/underwriting"}
approved==observed


## 20 — Identity drift

In [ ]:
observed2={**observed,"workload":"spiffe://prod.example/agents/unknown"}
{k:(approved.get(k),observed2.get(k)) for k in approved if approved.get(k)!=observed2.get(k)}


## 21 — Privilege drift

In [ ]:
approved_permissions={"submission.read"}
provisioned_permissions={"submission.read","submission.update"}
used_permissions={"submission.read"}
{"unapproved":provisioned_permissions-approved_permissions,
 "unused":provisioned_permissions-used_permissions}


## 22 — Posture score

In [ ]:
controls={"owner":True,"short_lived":True,"least_privilege":False,
"review_current":True,"monitoring":True,"runtime_binding":True}
score=round(100*sum(controls.values())/len(controls))
score


## 23 — Critical-control override

In [ ]:
critical={"static_prod_secret":False,"revoked_but_active":False,"audit_disabled":False}
effective_score=0 if any(critical.values()) else score
effective_score


## 24 — Scheduled recertification

In [ ]:
last_review=NOW-timedelta(days=70);interval=60
overdue=NOW>last_review+timedelta(days=interval)
overdue


## 25 — Event-driven review

In [ ]:
review_triggers={"owner_departed","new_write_tool","risk_increased","credential_compromised","new_external_trust"}
event="new_write_tool"
event in review_triggers


## 26 — Material change

In [ ]:
before={"actions":{"read"},"delegation":False,"external":False}
after={"actions":{"read","update"},"delegation":False,"external":False}
material=not after["actions"].issubset(before["actions"]) or after["delegation"]!=before["delegation"] or after["external"]!=before["external"]
material


## 27 — Exception record

In [ ]:
exception={"id":"EX-44","control":"no-static-prod-secrets","owner":"legacy-integrations",
"justification":"vendor migration","expires":NOW+timedelta(days=14),
"compensating_controls":["IP restriction","daily monitoring"]}
exception


## 28 — Exception expiry

In [ ]:
def exception_active(e,now=NOW):return now<e["expires"]
exception_active(exception,NOW+timedelta(days=20))


## 29 — Owner departure

In [ ]:
def owner_departure(identity):
    return {"identity":identity,"actions":["suspend new delegation","assign interim owner","trigger access review"]}
owner_departure("agt-001")


## 30 — Decommissioning

In [ ]:
decommission=["disable identity","revoke workload registration","revoke OAuth client",
"remove cloud role","revoke delegation","remove tool access","archive evidence","mark retired"]
pd.DataFrame({"step":range(1,len(decommission)+1),"action":decommission})


## 31 — Third-party agent record

In [ ]:
external={"id":"ext-21","provider":"ResearchCo","contract_owner":"procurement-ai",
"trust_domain":"research.example","approved_actions":["research.read"],
"incident_contact":"provider-soc","termination":"remove federation + revoke policy"}
external


## 32 — Supply-chain identity graph

In [ ]:
G=nx.DiGraph()
G.add_edges_from([("source:commit-42","build:991"),("build:991","artifact:sha256abc"),
("artifact:sha256abc","deployment:claims-v18"),("deployment:claims-v18","workload:spiffe-claims"),
("workload:spiffe-claims","agent:claims")])
list(nx.shortest_path(G,"source:commit-42","agent:claims"))


## 33 — Federation review

In [ ]:
federations=[{"trust_domain":"partner.example","last_review_days":20,"active":True},
{"trust_domain":"oldvendor.example","last_review_days":420,"active":True}]
[x for x in federations if x["active"] and x["last_review_days"]>180]


## 34 — Security event → governance action

In [ ]:
event_actions={"credential_compromised":"quarantine","owner_departed":"review",
"risk_increased":"reassess","federation_terminated":"revoke_external_trust"}
event_actions["credential_compromised"]


## 35 — Evidence bundle

In [ ]:
evidence={"identity":"agt-101","registration":"REG-101","risk":"medium",
"approvals":["owner","platform"],"policy_version":"identity-policy-v7",
"deployment":"sha256:abc","runtime_identity":"spiffe://prod.example/agents/underwriting",
"generated_at":NOW.isoformat()}
evidence


## 36 — Audit reconstruction

In [ ]:
audit=[
{"t":"09:00","event":"approved","actor":"platform-approver"},
{"t":"09:05","event":"provisioned","actor":"identity-controller"},
{"t":"09:10","event":"activated","actor":"deployment-controller"},
{"t":"11:42","event":"permission_change_requested","actor":"agent-owner"}]
pd.DataFrame(audit)


## 37 — Quarantine

In [ ]:
quarantine_effects={"new_sessions":False,"write_tools":False,"delegation":False,
"diagnostic_read":True,"forensic_state_preserved":True}
quarantine_effects


## 38 — Incident response

In [ ]:
incident=["detect","quarantine","revoke credentials","remove delegation",
"preserve evidence","analyze blast radius","remediate","re-attest","controlled recovery","postmortem"]
pd.DataFrame({"order":range(1,len(incident)+1),"phase":incident})


## 39 — KPI/KRI dataset

In [ ]:
metrics={"total_identities":1248,"high_risk":37,"unowned":3,"overprivileged":22,
"stale":15,"review_coverage_pct":98,"static_credentials":9,"mean_time_to_revoke_min":18}
pd.Series(metrics)


## 40 — Mean Time to Revoke

In [ ]:
detections=[("i1",4),("i2",22),("i3",11),("i4",35)]
sum(x[1] for x in detections)/len(detections)


## 41 — Identity debt

In [ ]:
debt={"static_keys":9,"shared_clients":4,"missing_owners":3,"overdue_reviews":7,
"overbroad_roles":22,"stale_federations":1}
sum(debt.values()),debt


## 42 — Dashboard priorities

In [ ]:
dashboard=pd.DataFrame([
{"finding":"static credential","count":9,"severity":"critical"},
{"finding":"overprivileged","count":22,"severity":"high"},
{"finding":"missing owner","count":3,"severity":"high"},
{"finding":"overdue review","count":7,"severity":"medium"}])
dashboard


## 43 — Governance control tests

In [ ]:
tests={
"critical_self_approval_blocked": not sod("alice",["alice"]),
"static_secret_deploy_blocked": not deployment_gate(bad,{"owner","platform"}),
"delegation_attenuated": request["actions"].issubset(delegation_policy["allowed_actions"]),
"runtime_binding": approved==observed,
"expired_exception_inactive": not exception_active(exception,NOW+timedelta(days=20))
}
tests


# 44 — Capstone: Enterprise Agent Identity Operating Model

Build a governed onboarding and operating workflow for a **high-risk claims agent**.

The agent:

```text
runs in production Kubernetes
uses SPIFFE workload identity
reads and updates claim records
calls two MCP tools
can spawn one short-lived read-only sub-agent
uses an external research agent
has no static credentials
```

Your system must:

```text
DISCOVER / REGISTER
        ↓
OWNER + PURPOSE
        ↓
RISK ASSESSMENT
        ↓
SECURITY + OWNER APPROVAL
        ↓
POLICY-AS-CODE GATE
        ↓
PROVISION WORKLOAD IDENTITY
        ↓
PROVISION ATTENUATED AUTHORITY
        ↓
BIND APPROVED DEPLOYMENT TO RUNTIME
        ↓
MONITOR IDENTITY + DELEGATION + DRIFT
        ↓
POSTURE / KPIs
        ↓
60-DAY RECERTIFICATION
        ↓
EVENT-DRIVEN REVIEW ON MATERIAL CHANGE
        ↓
QUARANTINE / REVOKE ON COMPROMISE
        ↓
RETIRE + ARCHIVE EVIDENCE
```

Adversarial tests:

1. developer attempts self-approval;
2. CI introduces a static production secret;
3. agent receives an unapproved write permission;
4. sub-agent requests broader authority than parent;
5. external trust domain becomes stale;
6. owner leaves;
7. exception expires;
8. runtime workload differs from approved deployment;
9. credential compromise occurs;
10. audit records are suppressed;
11. revoked identity attempts execution;
12. retirement leaves a cloud role behind.

The capstone is complete only if each failure produces a **machine-detectable control outcome**, not merely a written recommendation.


# Review questions

1. Why is an identity inventory more than a list of service accounts?
2. Which identities should be related to a logical agent?
3. Why must ownership be explicit?
4. What dimensions should influence agent identity risk?
5. How should approval vary by risk?
6. Why is segregation of duties important for identity governance?
7. How can identity policy become a CI/CD gate?
8. What is identity drift?
9. How does privilege drift differ from identity drift?
10. Why should posture scores have critical-control overrides?
11. What should recertification actually verify?
12. Which events should trigger immediate review?
13. What makes an exception governable?
14. What must happen when an agent is decommissioned?
15. How should third-party agent identity be governed?
16. How can software provenance strengthen runtime identity?
17. Why must federation relationships be reviewed?
18. What evidence should be retained for audit reconstruction?
19. What should Mean Time to Revoke measure?
20. What does governance-as-code enable that policy documents alone cannot?
